In [ ]:
!pip uninstall -y faiss-cpu faiss numpy
!pip install "numpy<2"         # installs 1.26.4 on Py3.10
!pip install faiss-cpu==1.8.0.post1

In [4]:
import numpy, faiss
print(numpy.__version__, faiss.__version__)

1.26.4 1.8.0


In [5]:
import os, ujson as json, numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss

/home/mmk2266/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
EMB_DIR = Path("data/rag_embeddings")
MODEL_ID = "Alibaba-NLP/gte-large-en-v1.5"   # same as used to embed
DIM = 1024

In [7]:
# ---------- Load embeddings + metadata ----------
embs = []
meta = []
for jf in sorted(EMB_DIR.glob("*.jsonl")):
    with jf.open("r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            vec = np.array(rec["embedding"], dtype="float32")
            embs.append(vec)
            meta.append({
                "id": rec.get("id"),
                "doc_id": rec.get("doc_id"),
                "type": rec.get("type"),
                "page": rec.get("page"),
                "section": rec.get("section"),
                "text": rec.get("text_for_embedding"),
                "image_path": rec.get("image_path"),
            })

In [8]:
embs = np.vstack(embs)           # shape: (N, 1024)  (already L2-normalized from your pipeline)
print("Loaded:", embs.shape, "vectors")

# ---------- Build FAISS (cosine via inner-product) ----------
index = faiss.IndexFlatIP(DIM)
index.add(embs)
print("FAISS added:", index.ntotal)

#save the FAISS index (with all vectors + internal structures) to disk once,
#so you can reload it instantly later
faiss.write_index(index, "data/rag.index")

Loaded: (5077, 1024) vectors
FAISS added: 5077


In [9]:
model = SentenceTransformer(MODEL_ID, trust_remote_code=True)

def search(query: str, k: int = 10):
    q = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    sims, ids = index.search(q, k)
    return sims[0], ids[0]

def show(query: str, k: int = 10, max_chars: int = 280):
    sims, ids = search(query, k)
    print(f"\nQuery: {query}\nTop {k} results:\n")
    for rank, (sid, score) in enumerate(zip(ids, sims), 1):
        m = meta[int(sid)]
        snippet = (m["text"] or "")[:max_chars].replace("\n", " ")
        print(f"{rank:>2}. score={score:.4f}  [{m['type']}]  {m['doc_id']}  p.{m['page']}  {m['section']}")
        print(f"    {snippet}")
        if m["type"] == "figure" and m.get("image_path"):
            print(f"    (image: {m['image_path']})")
        print()

In [10]:
show("training loss curves comparing AdamW vs SGD on CIFAR-10", k=10)


Query: training loss curves comparing AdamW vs SGD on CIFAR-10
Top 10 results:

 1. score=0.7345  [paragraph]  2203  p.9  Model and training details
    [SECTION] Model and training details [PAGE] 9 [PARAGRAPH] 8 Interestingly , a model trained with AdamW only passes the training performance of a model trained with Adam around 80% of the way through the cosine cycle, though the ending performance is notably better- see Figure A7

 2. score=0.7242  [paragraph]  2306  p.15  References
    [SECTION] References [PAGE] 15 [PARAGRAPH] Jingzhao Zhang, Sai Praneeth Karimireddy, Andreas Veit, Seungyeon Kim, Sashank J Reddi, Sanjiv Kumar, and Suvrit Sra. Why {adam} beats {sgd} for attention models, 2020.

 3. score=0.7030  [paragraph]  2203  p.9  Model and training details
    [SECTION] Model and training details [PAGE] 9 [PARAGRAPH] We use AdamW (Loshchilov and Hutter, 2019) for Chinchilla rather than Adam (Kingma and Ba, 2014) as this improves the language modelling loss and the downstream ta